# Recurrent Neural Networks (RNNs) Demonstration In Python

## Background

Think about Gmail Smart Compose—when you type “Kind…”, it suggests “Kind regards.” These systems work by learning patterns from large amounts of text, not by truly understanding language.

Email writing is highly repetitive, with common phrases like “thank you” or “looking forward,” which makes it ideal for pattern learning. 

Using an **LSTM**, the model remembers sequences of characters and learns what typically comes next. Instead of predicting whole words, it predicts one character at a time, making it flexible and capable of generating realistic email completions based on past patterns.


## Objective
Build a model that learns email writing patterns and auto-completes text by predicting the next character step-by-step.

## Process Flow

**1. Prepare data**
- Convert text → numbers

**2. Train model**
- Learn common patterns in email phrases

**3. Learn sequence**
- Predict next character based on previous ones

**4. Generate text**
- Start with input → keep predicting next characters

**5. Control output**
- Temperature decides safe vs creative text

### Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


### Data Preparation

In [2]:
# Email phrases dataset
sentences = [s.lower() for s in [
    "kind regards", "best regards", "warm regards", "with regards",
    "yours sincerely", "yours faithfully", "respectfully yours",
    "with best wishes", "best wishes for the week", "wishing you the best",
    "best wishes", "warmest wishes", "cheers", "talk soon",
    "see you soon", "see you tomorrow", "catch you later", "bye for now",
    "have a great day", "have a wonderful weekend", "enjoy your weekend",
    "take care", "stay safe", "hope this helps", "all the best",
    "thank you", "thanks so much", "thank you very much", "thanks a lot",
    "with gratitude", "many thanks", "thank you for your time",
    "thank you for your help", "thank you for your support",
    "thanks for reaching out", "thanks for the update", "thanks for the heads up",
    "looking forward to hearing from you", "looking forward to your reply",
    "let me know what you think", "let me know if you need anything",
    "hope to hear from you soon", "speak to you soon", "we will be in touch",
    "keep me posted", "keep me updated", "let us connect tomorrow",
    "please review the attachment", "please find attached",
    "hope you are doing well", "hope you are having a great week",
    "just checking in", "following up on our conversation",
    "sorry for the late reply", "sorry to bother you",
    "as per our discussion", "quick question for you",
    "sincerely", "regards", "cordially", "respectfully",
    "warmly", "gratefully", "faithfully",
    "thanks", "many thanks", "thank you", "warm regards",
    "best", "best regards", "kind regards", "with regards",
    "see you", "see you soon", "see you then", "see you there",
    "talk soon", "speak soon", "write back soon",
    "have a good day", "have a good one", "have a great weekend",
    "hope all is well", "hope you are well", "hope this works",
    "let me know", "please let me know", "do let me know",
    "looking forward to it", "looking forward to seeing you",
    "thanks again", "thank you again", "many thanks again",
    "take care", "cheers", "all the best", "warmly",
    "great working with you", "thanks for everything", "appreciate it",
    "much appreciated", "your help is appreciated", "thank you kindly",
    "wishing you well", "wishing you a great week", "wishing you success",
    "talk to you tomorrow", "catch up soon", "signing off"
]]

### Build character vocabulary
This step converts text into fixed-length numeric sequences and creates input–target pairs so the model learns to predict the next character.

In [3]:
# Build character-level vocabulary from all sentences
chars = sorted(set(" ".join(sentences)))   # unique characters (including space)

# Create mappings: character → index and index → character
char_to_ix = {c: i for i, c in enumerate(chars)}
ix_to_char = {i: c for c, i in char_to_ix.items()}

vocab_size = len(chars)  # total unique characters

# Find maximum sentence length for padding
max_len = max(len(s) for s in sentences)

# Convert sentence → numeric indices + pad with spaces to fixed length
def encode_and_pad(s):
    seq = [char_to_ix[c] for c in s]  # encode characters
    return seq + [char_to_ix[" "]] * (max_len - len(seq))  # right pad

# Prepare training data using sequence shifting:
# x = input sequence (all chars except last)
# y = target sequence (all chars except first)
x_data = torch.tensor([encode_and_pad(s)[:-1] for s in sentences])
y_data = torch.tensor([encode_and_pad(s)[1:] for s in sentences])

### Model

This model learns text patterns using LSTM and predicts the probability of the next character at each step.

In [4]:
# LSTM model for character-level text generation (auto-complete)
class EmailAutocompleteLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=32, hidden_size=256):
        super().__init__()
        
        # Converts character indices → dense vectors (learned representations)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # LSTM layer to capture sequential patterns in text
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        
        # Maps LSTM output → vocabulary space (predict next character)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)              # (batch, seq_len) → (batch, seq_len, embed_size)
        out, hidden = self.lstm(x, hidden) # sequence learning
        out = self.fc(out)                 # (batch, seq_len, vocab_size)
        return out, hidden                # raw scores (logits)

# Initialize model
model = EmailAutocompleteLSTM(vocab_size)

### Training

This step trains the model by comparing predictions with actual outputs and updating weights to minimize error.

In [5]:
# Loss function: compares predicted probabilities with actual next characters
criterion = nn.CrossEntropyLoss()

# Optimizer: updates model weights using Adam algorithm
optimizer = optim.Adam(model.parameters(), lr=0.003)

print(f"Vocab size: {vocab_size}")

# Training loop
for epoch in range(300):
    optimizer.zero_grad()  # reset gradients from previous step

    out, _ = model(x_data)  # forward pass (predictions)

    # Reshape to match CrossEntropy input: (N, vocab_size) vs (N,)
    loss = criterion(out.view(-1, vocab_size), y_data.view(-1))

    loss.backward()   # compute gradients
    optimizer.step() # update weights

    # Print loss every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:03d} | Loss: {loss.item():.4f}")

Vocab size: 25
Epoch 050 | Loss: 1.0888
Epoch 100 | Loss: 0.3774
Epoch 150 | Loss: 0.1025
Epoch 200 | Loss: 0.0714
Epoch 250 | Loss: 0.0648
Epoch 300 | Loss: 0.0624


### Generation Function

This function generates text by predicting one character at a time using the trained model and feeding each prediction back as input.

In [6]:
def autocomplete(model, seed, length=25, temperature=0.2):
    model.eval()  # set model to inference mode (no dropout)
    seed = seed.lower()

    # Ensure all characters exist in training vocabulary
    if any(c not in char_to_ix for c in seed):
        return "Invalid character in input."

    # Convert seed text → tensor of indices
    input_seq = torch.tensor([[char_to_ix[c] for c in seed]])
    output_text, hidden = seed, None

    with torch.no_grad():  # disable gradient computation
        # Pass full seed to initialize hidden state
        out, hidden = model(input_seq, hidden)
        out = out[:, -1, :]  # take last timestep output

        # Generate characters one by one
        for _ in range(length):
            # Apply temperature to control randomness
            probs = F.softmax(out / temperature, dim=-1)

            # Sample next character from probability distribution
            idx = torch.multinomial(probs, 1).item()
            char = ix_to_char[idx]

            output_text += char  # append to result

            # Feed predicted char back into model
            out, hidden = model(torch.tensor([[idx]]), hidden)
            out = out[:, -1, :]  # update for next prediction

    return output_text

### Testing

* **`t` (seed text)** → starting input given to the model (e.g., `"kind "`)
* **`l` (length)** → number of characters to generate
* **`temp` (temperature)** → controls randomness (low = safe, high = creative)


In [9]:

print("\n=== Predictions ===")
tests = [
    ("kind ", 12, 0.2),
    ("best ", 12, 0.2),
    ("many ", 12, 0.2),
    ("looki", 32, 0.2),
    ("hope ", 22, 0.3)
]

for t, l, temp in tests:
    print(f"{t} -> {autocomplete(model, t, l, temp)}")


=== Predictions ===
kind  -> kind regards     
best  -> best regards             
many  -> many thanks      
looki -> looking forward to hearing from you s
hope  -> hope you are well          
